# Preprocessing Device List
- The purpose of this notebook is to process the device list with *is_iot*,*device type*, and *vendor* label and also put them into a formatted structure

In [1]:
import os 
import sys
import pandas as pd
from typing import Dict, List
import re

In [2]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

In [3]:
DATASET_FOLDER = "../../../dataset"
DEVICE_LIST_1 = DATASET_FOLDER +"/identify-iot-devices-and-events-based-on-packet-length-from-encrypted-traffic/device-mapping/List_Of_Devices.txt"
DEVICE_LIST_2_FOLDER = DATASET_FOLDER + "/IoT-Dataset/devices"

sys.path.append(os.path.abspath(DEVICE_LIST_2_FOLDER))

from devices import devices
device_mapping_2 = devices
exported_folder = DATASET_FOLDER + "/processed-data"
device_mapping_folder = exported_folder + "/device-mapping"
input_data_folder = exported_folder + "/input-data"

## Identify IOT devices and events based on packet length dataset

### Device Mapping Preprocess

In [4]:
dataset1 = "dataset1"

In [5]:
MAC_ADDRESS_PATTERN = re.compile(r"[0-9a-fA-F]{2}(?::[0-9a-fA-F]{2}){5}")
IP_ADDRESS_PATTERN = re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")

DEVICE_LABELS = {
    "Smart Things": {"device_type": "hub", "vendor": "Samsung SmartThings"},
    "Amazon Echo": {"device_type": "smart speaker", "vendor": "Amazon"},
    "Netatmo Welcome": {"device_type": "camera", "vendor": "Netatmo"},
    "TP-Link Day Night Cloud camera": {"device_type": "camera", "vendor": "TP-Link"},
    "Samsung SmartCam": {"device_type": "camera", "vendor": "Samsung"},
    "Dropcam": {"device_type": "camera", "vendor": "Dropcam"},
    "Insteon Camera": {"device_type": "camera", "vendor": "Insteon"},
    "Withings Smart Baby Monitor": {"device_type": "baby monitor", "vendor": "Withings"},
    "Belkin Wemo switch": {"device_type": "smart switch", "vendor": "Belkin"},
    "TP-Link Smart plug": {"device_type": "smart plug", "vendor": "TP-Link"},
    "iHome": {"device_type": "smart plug", "vendor": "iHome"},
    "Belkin wemo motion sensor": {"device_type": "motion sensor", "vendor": "Belkin"},
    "NEST Protect smoke alarm": {"device_type": "smoke alarm", "vendor": "Nest"},
    "Netatmo weather station": {"device_type": "weather station", "vendor": "Netatmo"},
    "Withings Smart scale": {"device_type": "smart scale", "vendor": "Withings"},
    "Blipcare Blood Pressure meter": {"device_type": "blood pressure monitor", "vendor": "Blipcare"},
    "Withings Aura smart sleep sensor": {"device_type": "sleep sensor", "vendor": "Withings"},
    "Light Bulbs LiFX Smart Bulb": {"device_type": "smart bulb", "vendor": "LIFX"},
    "Triby Speaker": {"device_type": "smart speaker", "vendor": "Invoxia"},
    "PIX-STAR Photo-frame": {"device_type": "photo frame", "vendor": "Pix-Star"},
    "Nest Dropcam": {"device_type": "camera", "vendor": "Nest"},
    "HP Printer": {"device_type": "printer", "vendor": "HP"},
}

DEVICE_MAPPING_COLUMNS = [
    "device_name",
    "ip_address",
    "mac_address",
    "is_iot_device",
    "device_type",
    "vendor",
]


def clean_device_name(raw_name: str) -> str:
    return re.sub(r"\s+", " ", raw_name).strip()


def parse_list_of_devices(device_list_path: str) -> pd.DataFrame:
    records = []

    with open(device_list_path, "r", encoding="utf-8") as device_list_file:
        for line_number, raw_line in enumerate(device_list_file, start=1):
            line = raw_line.strip()

            if not line or line.lower().startswith("list of devices"):
                continue

            mac_match = MAC_ADDRESS_PATTERN.search(line)
            if mac_match is None:
                raise ValueError(f"Could not find a MAC address on line {line_number}: {raw_line!r}")

            device_name = clean_device_name(line[:mac_match.start()])
            mac_address = mac_match.group(0).lower()
            ip_match = IP_ADDRESS_PATTERN.search(line)
            ip_address = ip_match.group(0) if ip_match else ""

            iot_label = DEVICE_LABELS.get(device_name)
            if iot_label is not None:
                records.append(
                    {
                        "device_name": device_name,
                        "ip_address": ip_address,
                        "mac_address": mac_address,
                        "is_iot_device": 1,
                        "device_type": iot_label["device_type"],
                        "vendor": iot_label["vendor"],
                    }
                )
            else:
                records.append({
                    "device_name": device_name,
                    "ip_address": ip_address,
                    "mac_address": mac_address,
                    "is_iot_device": 0,
                    "device_type": None,
                    "vendor": None,
                })

    device_mapping = pd.DataFrame(records, columns=DEVICE_MAPPING_COLUMNS, dtype=object)
    return device_mapping


device_mapping_1 = parse_list_of_devices(DEVICE_LIST_1)
dataset1_device_mapping_folder = device_mapping_folder + "/" + dataset1
os.makedirs(dataset1_device_mapping_folder, exist_ok=True)
device_mapping_1_output_path = os.path.join(dataset1_device_mapping_folder, "list_of_devices_processed.csv")
device_mapping_1.to_csv(device_mapping_1_output_path, index=False, na_rep="None")

device_mapping_1

,device_name,ip_address,mac_address,is_iot_device,device_type,vendor
0,Smart Things,,d0:52:a8:00:67:5e,1,hub,Samsung SmartThings
1,Amazon Echo,,44:65:0d:56:cc:d3,1,smart speaker,Amazon
2,Netatmo Welcome,,70:ee:50:18:34:43,1,camera,Netatmo
3,TP-Link Day Night Cloud camera,,f4:f2:6d:93:51:f1,1,camera,TP-Link
4,Samsung SmartCam,,00:16:6c:ab:6b:88,1,camera,Samsung
5,Dropcam,,30:8c:fb:2f:e4:b2,1,camera,Dropcam
6,Insteon Camera,,00:62:6e:51:27:2e,1,camera,Insteon
7,Insteon Camera,,e8:ab:fa:19:de:4f,1,camera,Insteon
8,Withings Smart Baby Monitor,,00:24:e4:11:18:a8,1,baby monitor,Withings
9,Belkin Wemo switch,,ec:1a:59:79:f4:89,1,smart switch,Belkin


### Input Dataset Labeling

```bash
for f in *.pcap; do
    base="${f%.pcap}"
    echo -e "Processing file ${f}...."
    tshark -n \
    -r ${f} \
    -T fields \
    -E header=y \
    -E separator=, \
    -E quote=d \
    -E occurrence=f \
    -e frame.number \
    -e frame.time_epoch \
    -e frame.time_relative \
    -e frame.time_delta \
    -e frame.cap_len \
    -e frame.len \
    -e frame.interface_id \
    -e frame.encap_type \
    -e frame.protocols \
    -e eth.src \
    -e eth.dst \
    -e eth.src.oui \
    -e eth.dst.oui \
    -e eth.type \
    -e eth.len \
    -e eth.padding \
    -e eth.trailer \
    -e eth.fcs \
    -e eth.fcs.status \
    -e vlan.id \
    -e vlan.priority \
    -e vlan.etype \
    -e arp.opcode \
    -e arp.src.hw_mac \
    -e arp.src.proto_ipv4 \
    -e arp.dst.hw_mac \
    -e arp.dst.proto_ipv4 \
    -e ip.version \
    -e ip.hdr_len \
    -e ip.dsfield \
    -e ip.len \
    -e ip.id \
    -e ip.flags \
    -e ip.flags.df \
    -e ip.flags.mf \
    -e ip.frag_offset \
    -e ip.ttl \
    -e ip.proto \
    -e ip.checksum \
    -e ip.checksum.status \
    -e ip.src \
    -e ip.dst \
    -e ipv6.version \
    -e ipv6.tclass \
    -e ipv6.flow \
    -e ipv6.plen \
    -e ipv6.nxt \
    -e ipv6.hlim \
    -e ipv6.src \
    -e ipv6.dst \
    -e icmp.type \
    -e icmp.code \
    -e icmp.checksum \
    -e icmpv6.type \
    -e icmpv6.code \
    -e igmp.type \
    -e igmp.version \
    > ./csv_exported/${base}.csv
done
```

## IoT-Dataset
- One problem is that multiple devices are connecting through a hub

```bash
for f in *.pcap; do
    base="${f%.pcap}"
    echo -e "Processing file ${f}...."
    tshark -n \
    -r ${f} \
    -T fields \
    -E header=y \
    -E separator=, \
    -E quote=d \
    -E occurrence=f \
    -e frame.number \
    -e frame.time_epoch \
    -e frame.time_relative \
    -e frame.time_delta \
    -e frame.cap_len \
    -e frame.len \
    -e frame.interface_id \
    -e frame.encap_type \
    -e frame.protocols \
    -e eth.src \
    -e eth.dst \
    -e eth.src.oui \
    -e eth.dst.oui \
    -e eth.type \
    -e eth.len \
    -e eth.padding \
    -e eth.trailer \
    -e eth.fcs \
    -e eth.fcs.status \
    -e vlan.id \
    -e vlan.priority \
    -e vlan.etype \
    -e arp.opcode \
    -e arp.src.hw_mac \
    -e arp.src.proto_ipv4 \
    -e arp.dst.hw_mac \
    -e arp.dst.proto_ipv4 \
    -e ip.version \
    -e ip.hdr_len \
    -e ip.dsfield \
    -e ip.len \
    -e ip.id \
    -e ip.flags \
    -e ip.flags.df \
    -e ip.flags.mf \
    -e ip.frag_offset \
    -e ip.ttl \
    -e ip.proto \
    -e ip.checksum \
    -e ip.checksum.status \
    -e ip.src \
    -e ip.dst \
    -e ipv6.version \
    -e ipv6.tclass \
    -e ipv6.flow \
    -e ipv6.plen \
    -e ipv6.nxt \
    -e ipv6.hlim \
    -e ipv6.src \
    -e ipv6.dst \
    -e icmp.type \
    -e icmp.code \
    -e icmp.checksum \
    -e icmpv6.type \
    -e icmpv6.code \
    -e igmp.type \
    -e igmp.version \
    > ./csv_exported/${base}.csv
done
```

In [6]:
dataset2 = "dataset2"

DEVICE_MAPPING_2_COLUMNS = [
    "device_name",
    "ip_address",
    "mac_address",
    "is_iot_device",
    "device_type",
    "vendor",
]

DEVICE_MAPPING_2_LABELS = {
    "arlo-hub-01": {"device_type": "hub", "vendor": "Arlo"},
    "nest-cam-01": {"device_type": "camera", "vendor": "Nest"},
    "nest-doorbell-01": {"device_type": "video doorbell", "vendor": "Nest"},
    "amazon-show-01": {"device_type": "smart display", "vendor": "Amazon"},
    "amazon-dot-01": {"device_type": "smart speaker", "vendor": "Amazon"},
    "amazon-dot-02": {"device_type": "smart speaker", "vendor": "Amazon"},
    "geeni-cam-01": {"device_type": "camera", "vendor": "Geeni"},
    "geeni-cam-02": {"device_type": "camera", "vendor": "Geeni"},
    "geeni-cam-03": {"device_type": "camera", "vendor": "Geeni"},
    "ring-hub-01": {"device_type": "hub", "vendor": "Ring"},
    "smartthings-hub-01": {"device_type": "hub", "vendor": "Samsung SmartThings"},
    "hue-hub-01": {"device_type": "hub", "vendor": "Philips Hue"},
    "kasa-cam-01": {"device_type": "camera", "vendor": "TP-Link Kasa"},
    "nightowl-doorbell-01": {"device_type": "video doorbell", "vendor": "Night Owl"},
    "google-home-01": {"device_type": "smart speaker", "vendor": "Google"},
    "ring-doorbell-01": {"device_type": "video doorbell", "vendor": "Ring"},
    "ring-doorbell-02": {"device_type": "video doorbell", "vendor": "Ring"},
    "ring-doorbell-03": {"device_type": "video doorbell", "vendor": "Ring"},
    "schlage-lock-01": {"device_type": "smart lock", "vendor": "Schlage"},
    "august-hub-01": {"device_type": "hub", "vendor": "August"},
    "ultraloq-hub-01": {"device_type": "hub", "vendor": "Ultraloq"},
    "blink-cam-03": {"device_type": "camera", "vendor": "Blink"},
    "sifely-hub-01": {"device_type": "hub", "vendor": "Sifely"},
    "lockly-hub-01": {"device_type": "hub", "vendor": "Lockly"},
    "blink-cam-01": {"device_type": "camera", "vendor": "Blink"},
    "blink-cam-02": {"device_type": "camera", "vendor": "Blink"},
    "ring-chime-01": {"device_type": "chime", "vendor": "Ring"},
    "simplisafe-d1": {"device_type": "video doorbell", "vendor": "SimpliSafe"},
    "simplisafe-d2": {"device_type": "camera", "vendor": "SimpliSafe"},
    "smartthings-cam-01": {"device_type": "camera", "vendor": "Samsung SmartThings"},
    "merkury-cam-01": {"device_type": "camera", "vendor": "Merkury Innovations"},
    "wyze-cam-01": {"device_type": "camera", "vendor": "Wyze"},
    "geeni-doorbell-02": {"device_type": "video doorbell", "vendor": "Geeni"},
    "nightowl-doorbell-02": {"device_type": "video doorbell", "vendor": "Night Owl"},
    "ring-light-01": {"device_type": "camera", "vendor": "Ring"},
    "merkury-doorbell-01": {"device_type": "video doorbell", "vendor": "Merkury Innovations"},
    "geeni-doorbell-01": {"device_type": "video doorbell", "vendor": "Geeni"},
    "august-hub-02": {"device_type": "hub", "vendor": "August"},
    "blink-hub-01": {"device_type": "hub", "vendor": "Blink"},
    "august-lock-01": {"device_type": "smart lock", "vendor": "August"},
    "sifely-lock-01": {"device_type": "smart lock", "vendor": "Sifely"},
    "lockly-lock-01": {"device_type": "smart lock", "vendor": "Lockly"},
    "ultraloq-lock-01": {"device_type": "smart lock", "vendor": "Ultraloq"},
    "yale-lock-02": {"device_type": "smart lock", "vendor": "Yale"},
    "smartthings-multi-01": {"device_type": "multipurpose sensor", "vendor": "Samsung SmartThings"},
    "smartthings-water-01": {"device_type": "water leak sensor", "vendor": "Samsung SmartThings"},
    "smartthings-motion-01": {"device_type": "motion sensor", "vendor": "Samsung SmartThings"},
    "smartthings-light-01": {"device_type": "smart light", "vendor": "Samsung SmartThings"},
    "hue-light-01": {"device_type": "smart light", "vendor": "Philips Hue"},
    "hue-light-02": {"device_type": "smart light", "vendor": "Philips Hue"},
    "hue-light-03": {"device_type": "smart light", "vendor": "Philips Hue"},
    "hue-light-04": {"device_type": "smart light", "vendor": "Philips Hue"},
    "hue-light-05": {"device_type": "smart light", "vendor": "Philips Hue"},
    "yale-lock-01": {"device_type": "smart lock", "vendor": "Yale"},
    "ring-keypad-01": {"device_type": "keypad", "vendor": "Ring"},
    "ring-keypad-02": {"device_type": "keypad", "vendor": "Ring"},
    "ring-contact-sensor-01": {"device_type": "contact sensor", "vendor": "Ring"},
    "ring-contact-sensor-02": {"device_type": "contact sensor", "vendor": "Ring"},
    "ring-contact-sensor-03": {"device_type": "contact sensor", "vendor": "Ring"},
    "ring-motion-01": {"device_type": "motion sensor", "vendor": "Ring"},
    "ring-motion-02": {"device_type": "motion sensor", "vendor": "Ring"},
    "ring-motion-03": {"device_type": "motion sensor", "vendor": "Ring"},
    "ring-range-01": {"device_type": "range extender", "vendor": "Ring"},
    "arlo-cam-01": {"device_type": "camera", "vendor": "Arlo"},
    "arlo-cam-02": {"device_type": "camera", "vendor": "Arlo"},
    "smartthings-button-01": {"device_type": "button", "vendor": "Samsung SmartThings"},
    "smartthings-arrival-01": {"device_type": "button", "vendor": "Samsung SmartThings"},
    "smartthings-tracker-01": {"device_type": "tracker", "vendor": "Samsung SmartThings"},
    "geeni-awarecam-1": {"device_type": "camera", "vendor": "Geeni"},
    "geeni-awarecam-2": {"device_type": "camera", "vendor": "Geeni"},
    "kwikset-lock-01": {"device_type": "smart lock", "vendor": "Kwikset"},
    "smartthings-outlet-01": {"device_type": "smart outlet", "vendor": "Samsung SmartThings"},
}


def clean_device_mapping_2_value(value: object) -> str:
    if value is None:
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()


def transform_device_mapping_2(mapping: Dict[str, Dict[str, str]]) -> pd.DataFrame:
    missing_labels = sorted(set(mapping) - set(DEVICE_MAPPING_2_LABELS))
    if missing_labels:
        raise ValueError(f"Missing device labels for: {missing_labels}")

    records = []

    for device_id, device_info in mapping.items():
        label = DEVICE_MAPPING_2_LABELS[device_id]
        if ((clean_device_mapping_2_value(device_info.get("address", ""))
            or clean_device_mapping_2_value(device_info.get("ether", "")).lower())): 
            records.append(
                {
                    "device_name": device_id,
                    "ip_address": clean_device_mapping_2_value(device_info.get("address", "")),
                    "mac_address": clean_device_mapping_2_value(device_info.get("ether", "")).lower(),
                    "is_iot_device": 1,
                    "device_type": label["device_type"],
                    "vendor": label["vendor"],
                }
            )

    device_mapping = pd.DataFrame(records, columns=DEVICE_MAPPING_2_COLUMNS, dtype=object)
    return device_mapping


device_mapping_2_processed = transform_device_mapping_2(device_mapping_2)
dataset2_device_mapping_folder = device_mapping_folder + "/" + dataset2
os.makedirs(dataset2_device_mapping_folder, exist_ok=True)

device_mapping_2_output_path = os.path.join(dataset2_device_mapping_folder, "list_of_devices_processed.csv")
device_mapping_2_processed.to_csv(device_mapping_2_output_path, index=False, na_rep="None")

device_mapping_2_processed

,device_name,ip_address,mac_address,is_iot_device,device_type,vendor
0,arlo-hub-01,192.168.1.100,9c:3d:cf:94:aa:c3,1,hub,Arlo
1,nest-cam-01,192.168.1.101,18:b4:30:8f:ec:23,1,camera,Nest
2,nest-doorbell-01,192.168.1.102,64:16:66:73:e6:e0,1,video doorbell,Nest
3,amazon-show-01,192.168.1.104,4c:17:44:a4:0b:30,1,smart display,Amazon
4,amazon-dot-01,192.168.1.105,08:84:9d:09:4a:69,1,smart speaker,Amazon
5,amazon-dot-02,192.168.1.106,fc:49:2d:76:33:a5,1,smart speaker,Amazon
6,geeni-cam-01,192.168.1.107,58:b3:fc:68:a6:e2,1,camera,Geeni
7,geeni-cam-02,192.168.1.108,0c:8c:24:72:71:9a,1,camera,Geeni
8,geeni-cam-03,192.168.1.109,0c:8c:24:7f:34:84,1,camera,Geeni
9,ring-hub-01,192.168.1.110,b0:09:da:12:90:23,1,hub,Ring
